In [2]:
import sys
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

# Directory paths
ROOT_DIR = Path.cwd()
SRC_DIR = ROOT_DIR / "src"
DATA_DIR = ROOT_DIR / "data"


def get_jsonl_files():
    return sorted(DATA_DIR.glob("*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)

In [3]:
# Run the scraper script in src/
scraper_files = sorted(SRC_DIR.glob("*.py"))
if not scraper_files:
    raise FileNotFoundError("No Python files found in 'src/'.")

for script in scraper_files:
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True,
        text=True
    )
    print(f"Ran: {script.name}")
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")

# Read the latest JSONL file as pandas DataFrame
jsonl_files = get_jsonl_files()
if not jsonl_files:
    raise FileNotFoundError("No JSONL file found in 'data/' after running scraper.")

jsonl_path = jsonl_files[0]
data = pd.read_json(jsonl_path, lines=True)

print(f"Loaded: {jsonl_path}")
data

Ran: edcs_scraper.py
[+] Connecting to EDCS API...
[+] Connected. Page size 500 works. Total in EDCS: 542,296

[lookup] No changes — using existing lookup file.

[+] Data files found. Checking for updates...
    Local monuments : 542,296
    EDCS total      : 542,296

[✓] No new records found — local data is up to date.
[✓] Local monuments : 542,296
[✓] EDCS total      : 542,296

[✓] JSONL  : /Users/sanoj/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.jsonl
[✓] TSV    : /Users/sanoj/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.tsv
[✓] Lookup : /Users/sanoj/Documents/Projects/EDCS-Analytics/data/edcs_lookup.json

Loaded: /Users/sanoj/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.jsonl


,record_id,edcs_id,inscription_index,province,place,latitude,longitude,material,material_en,not_before,not_after,inscription_text,language,category,category_en,belege,image_urls
0,EDCS-00000001-0,EDCS-00000001,0,Noricum,Maribor / Marburg,46.555628,15.644770,lapis,stone,,,?] C(aius) Trebonius IIvir et praef(ectus) i(u...,la,"[tituli sepulcrales, viri, praenomen et nomen,...","[tomb inscriptions, men, first name and genti...",[ILSlov-02-01 0 00455],bilder/Mu/Muchar_1_p_398.jpg
1,EDCS-00000002-0,EDCS-00000002,0,NaN,NaN,NaN,NaN,lapis,stone,,,]ITSAMARVS[3] / [3]ETS TOVICTO[,la,[tituli sacri],[dedicatory inscriptions],[Noelke-2021 00040],bilder/No/Noelke-2021_00040.jpg
2,EDCS-00000003-0,EDCS-00000003,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,lapis,stone,,,[I(ovi)] O(ptimo) M(aximo),la,[tituli sacri],[dedicatory inscriptions],[Zimmermann-2022 0 p 19],bilder/Zi/Zimmermann-2022_19.jpg
3,EDCS-00000004-0,EDCS-00000004,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,opus figlinae,ceramics,,,L(egio) I M(inervia),la,[tituli fabricationis],[manufacturer's inscriptions],[Zimmermann-2022 0 p 20],
4,EDCS-00000005-0,EDCS-00000005,0,Venetia et Histria / Regio X,Este / Ateste,45.224007,11.659796,aes,bronze,,,C(aius) Artilius,la,"[sigilla impressa, praenomen et nomen]","[stamped inscriptions, first name and gentile...",[ArchVen 2022 145],bilder/Ar/ArchVen-2022-145.jpg | bilder/Ar/Arc...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587889,EDCS-85701223-1,EDCS-85701223,1,Baetica,Almensilla,37.310001,-6.113052,lapis,stone,508,508,Hispalensis fa/mulus dei vixit / ann(os) plus ...,la,"[tituli sepulcrales, inscriptiones christianae...","[tomb inscriptions, christian inscriptions, me...",[FE 00898],bilder/FE/FE_00898.jpg
587890,EDCS-85701224-0,EDCS-85701224,0,Hispania citerior,Banos de Rio Caldo,41.856406,-8.107254,opus figlinae,ceramics,,,[3]SCI / [3 O]gere(n)sibu[s,la,[tituli possessionis],[owner inscriptions],[FE 00899],bilder/FE/FE_00899.jpg
587891,EDCS-85701225-0,EDCS-85701225,0,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,97,[Imp(erator) Ne]rva Ca[esar A]ugust[us pontife...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...","[CIL 16 00041, IScM 05 00291, RMD 05 p 699, AE...",
587892,EDCS-85701225-1,EDCS-85701225,1,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,97,quae est Lyciae et Pamphylia]e sub Iulio Mar[i...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...","[CIL 16 00041, IScM 05 00291, RMD 05 p 699, AE...",


In [4]:
list(data.columns)

['record_id',
 'edcs_id',
 'inscription_index',
 'province',
 'place',
 'latitude',
 'longitude',
 'material',
 'material_en',
 'not_before',
 'not_after',
 'inscription_text',
 'language',
 'category',
 'category_en',
 'belege',
 'image_urls']

In [5]:
if "belege" in data.columns:
    data.rename(columns={"belege": "evidence"}, inplace=True)

In [6]:
import re

# This cell updates the existing `data` DataFrame in-memory only.
# It does NOT read from or write to JSON/JSONL files.

def step1_dubious_dot(text: str) -> str:
    # 1) Remove dubious dot subscript (U+0323) FIRST
    return text.replace("\u0323", "")


def step2_edcs_gaps(text: str) -> str:
    # 2) EDCS gaps: [3]/[6] -> [-], [1] -> space
    text = re.sub(r"\[6\]", "[-]", text)
    text = re.sub(r"\[3\]", "[-]", text)
    text = re.sub(r"\[1\]", " ", text)
    return text


def step3_quotes_backslashes(text: str) -> str:
    # 3) Remove quotes and backslashes
    return text.replace("\\", "").replace('"', "").replace("'", "")


def step4_conservative(text: str) -> str:
    # 4) Conservative editorial handling
    text = re.sub(r"\([^)]*\)", "", text)          # remove (abc) incl. contents
    text = re.sub(r"\{([^}]*)\}", r"\1", text)      # keep {abc} contents, drop braces
    text = re.sub(r"\[[^\]]*\]", "", text)        # remove [abc] incl. contents
    text = re.sub(r"<([^=>]*)=[^>]*>", r"\1", text)  # <a=b> -> a
    text = re.sub(r"<[^>]*>", "", text)             # remove other <abc> incl. contents
    return text


def step4_interpretive(text: str) -> str:
    # 4) Interpretive editorial handling
    text = re.sub(r"\(([^)]*)\)", r"\1", text)      # keep (abc) contents
    text = re.sub(r"\[([^\]]*)\]", r"\1", text)    # keep [abc] contents
    text = re.sub(r"\{[^}]*\}", "", text)          # remove {abc} incl. contents
    text = re.sub(r"<[^=><]*=([^>]*)>", r"\1", text) # <a=b> -> b
    text = re.sub(r"<([^>]*)>", r"\1", text)         # keep other <abc> contents
    return text


def step5_line_breaks(text: str) -> str:
    # 5) Replace / with space
    return text.replace("/", " ")


def step6_punctuation_symbols(text: str) -> str:
    # 6) Remove punctuation and epigraphic symbols
    text = re.sub(r"[,\.\-\u2014:;!#%\^&~@]", "", text)
    text = re.sub(r"[\u2766\u00b7\u2219]", "", text)  # ❦ · ∙
    return text


def step7_uncertainty(text: str) -> str:
    # 7) Remove ?
    return text.replace("?", "")


def step8_arabic_numerals(text: str) -> str:
    # 8) Remove Arabic numerals
    return re.sub(r"[0-9]", "", text)


def step9_unclosed_brackets(text: str) -> str:
    # 9) Remove leftover bracket chars
    return re.sub(r"[\[\]\{\}()]", "", text)


def step10_que_enclitic(text: str) -> str:
    # 10) Separate enclitic -que: libertatisque -> libertatis que
    return re.sub(r"(?<=[A-Za-z])(que)(?=\s|$)", r" \1", text)


def step11_numeral_vir(text: str) -> str:
    # 11) Roman numeral + vir: IIIIviribus -> IIII viribus
    return re.sub(r"([IVXLCDMivxlcdm]+)(vir\w*)", r"\1 \2", text)


def step12_collapse_spaces(text: str) -> str:
    # 12) Collapse multiple spaces
    return re.sub(r"\s+", " ", text)


def step13_strip(text: str) -> str:
    # 13) Strip leading/trailing whitespace LAST
    return text.strip()


def clean_conservative(raw: str) -> str:
    t = raw
    t = step1_dubious_dot(t)
    t = step2_edcs_gaps(t)
    t = step3_quotes_backslashes(t)
    t = step4_conservative(t)
    t = step5_line_breaks(t)
    t = step6_punctuation_symbols(t)
    t = step7_uncertainty(t)
    t = step8_arabic_numerals(t)
    t = step9_unclosed_brackets(t)
    t = step10_que_enclitic(t)
    t = step11_numeral_vir(t)
    t = step12_collapse_spaces(t)
    t = step13_strip(t)
    return t


def clean_interpretive(raw: str) -> str:
    t = raw
    t = step1_dubious_dot(t)
    t = step2_edcs_gaps(t)
    t = step3_quotes_backslashes(t)
    t = step4_interpretive(t)
    t = step5_line_breaks(t)
    t = step6_punctuation_symbols(t)
    t = step7_uncertainty(t)
    t = step8_arabic_numerals(t)
    t = step9_unclosed_brackets(t)
    t = step10_que_enclitic(t)
    t = step11_numeral_vir(t)
    t = step12_collapse_spaces(t)
    t = step13_strip(t)
    return t


# Keep raw inscription_text unchanged; compute new columns from a safe copy
raw_series = data["inscription_text"].fillna("").astype(str)

data["inscription_text_conservative"] = raw_series.map(clean_conservative)
data["inscription_text_interpretive"] = raw_series.map(clean_interpretive)
data["is_unreadable"] = raw_series.map(lambda s: s.strip() in ("", "?"))
data["is_forged"] = data["evidence"].fillna("").astype(str).str.contains("*", regex=False)

data

,record_id,edcs_id,inscription_index,province,place,latitude,longitude,material,material_en,not_before,...,inscription_text,language,category,category_en,evidence,image_urls,inscription_text_conservative,inscription_text_interpretive,is_unreadable,is_forged
0,EDCS-00000001-0,EDCS-00000001,0,Noricum,Maribor / Marburg,46.555628,15.644770,lapis,stone,,...,?] C(aius) Trebonius IIvir et praef(ectus) i(u...,la,"[tituli sepulcrales, viri, praenomen et nomen,...","[tomb inscriptions, men, first name and genti...",[ILSlov-02-01 0 00455],bilder/Mu/Muchar_1_p_398.jpg,C Trebonius II vir et praef i d civitatis Agunti,Caius Trebonius II vir et praefectus iure dicu...,False,False
1,EDCS-00000002-0,EDCS-00000002,0,NaN,NaN,NaN,NaN,lapis,stone,,...,]ITSAMARVS[3] / [3]ETS TOVICTO[,la,[tituli sacri],[dedicatory inscriptions],[Noelke-2021 00040],bilder/No/Noelke-2021_00040.jpg,ITSAMARVS ETS TOVICTO,ITSAMARVS ETS TOVICTO,False,False
2,EDCS-00000003-0,EDCS-00000003,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,lapis,stone,,...,[I(ovi)] O(ptimo) M(aximo),la,[tituli sacri],[dedicatory inscriptions],[Zimmermann-2022 0 p 19],bilder/Zi/Zimmermann-2022_19.jpg,O M,Iovi Optimo Maximo,False,False
3,EDCS-00000004-0,EDCS-00000004,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,opus figlinae,ceramics,,...,L(egio) I M(inervia),la,[tituli fabricationis],[manufacturer's inscriptions],[Zimmermann-2022 0 p 20],,L I M,Legio I Minervia,False,False
4,EDCS-00000005-0,EDCS-00000005,0,Venetia et Histria / Regio X,Este / Ateste,45.224007,11.659796,aes,bronze,,...,C(aius) Artilius,la,"[sigilla impressa, praenomen et nomen]","[stamped inscriptions, first name and gentile...",[ArchVen 2022 145],bilder/Ar/ArchVen-2022-145.jpg | bilder/Ar/Arc...,C Artilius,Caius Artilius,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587889,EDCS-85701223-1,EDCS-85701223,1,Baetica,Almensilla,37.310001,-6.113052,lapis,stone,508,...,Hispalensis fa/mulus dei vixit / ann(os) plus ...,la,"[tituli sepulcrales, inscriptiones christianae...","[tomb inscriptions, christian inscriptions, me...",[FE 00898],bilder/FE/FE_00898.jpg,Hispalensis fa mulus dei vixit ann plus minus ...,Hispalensis fa mulus dei vixit annos plus minu...,False,False
587890,EDCS-85701224-0,EDCS-85701224,0,Hispania citerior,Banos de Rio Caldo,41.856406,-8.107254,opus figlinae,ceramics,,...,[3]SCI / [3 O]gere(n)sibu[s,la,[tituli possessionis],[owner inscriptions],[FE 00899],bilder/FE/FE_00899.jpg,SCI geresibus,SCI Ogerensibus,False,False
587891,EDCS-85701225-0,EDCS-85701225,0,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,...,[Imp(erator) Ne]rva Ca[esar A]ugust[us pontife...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...","[CIL 16 00041, IScM 05 00291, RMD 05 p 699, AE...",,rva Caugust tribunestate ctibus us q Flavi et ...,Imperator Nerva Caesar Augustus pontifex maxim...,False,False
587892,EDCS-85701225-1,EDCS-85701225,1,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,...,quae est Lyciae et Pamphylia]e sub Iulio Mar[i...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...","[CIL 16 00041, IScM 05 00291, RMD 05 p 699, AE...",,quae est Lyciae et Pamphyliae sub Iulio Marbsc...,quae est Lyciae et Pamphyliae sub Iulio Marino...,False,False


In [7]:
forged = int(data["is_forged"].sum())
unreadable = int(data["is_unreadable"].sum())
print(f"Forged: {forged}\nUnreadable: {unreadable}")

Forged: 6263
Unreadable: 13209


In [8]:
raw = data["inscription_text"].fillna("").astype(str).str.strip()
cons = data["inscription_text_conservative"].fillna("").astype(str).str.strip()
interp = data["inscription_text_interpretive"].fillna("").astype(str).str.strip()

raw_nonempty_mask = raw.ne("")

raw_inscriptions = int((raw_nonempty_mask & raw.eq("")).sum())
cons_became_empty = int((raw_nonempty_mask & cons.eq("")).sum())
interp_became_empty = int((raw_nonempty_mask & interp.eq("")).sum())

summary = pd.DataFrame({
    "before pd.NA": [raw_inscriptions, cons_became_empty, interp_became_empty],
}, index=["raw inscription","conservative", "interpretive"])

summary["after pd.NA"] = [
    int(raw.eq("").sum()),
    int(cons.eq("").sum()),
    int(interp.eq("").sum()),
]

summary


,before pd.NA,after pd.NA
raw inscription,0,140
conservative,13784,13924
interpretive,13464,13604


In [9]:
# Handling empty data points to recognize NULL values
missing_counts = data.isna().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

latitude      20618
longitude     20618
province       3950
place          3950
not_after        35
not_before       22
dtype: int64

In [10]:
data = data.replace(r'^\s*$', pd.NA, regex=True)

In [11]:
null_info = data.isna().sum()
null_columns = pd.DataFrame({
    "null_count": null_info.astype("int64"),
    "rows": len(data),
})

null_columns["null_pct"] = (null_columns["null_count"] / len(data) * 100).round(2)

null_columns[null_columns["null_count"] > 0].sort_values("null_count", ascending=False)

,null_count,rows,null_pct
image_urls,463053,587894,78.76
not_after,368658,587894,62.71
not_before,368645,587894,62.71
material,81806,587894,13.92
material_en,81806,587894,13.92
latitude,20618,587894,3.51
longitude,20618,587894,3.51
inscription_text_conservative,13924,587894,2.37
inscription_text_interpretive,13604,587894,2.31
province,3950,587894,0.67
